# 3. Snowflake: Import the Databricks Ossie from S3

This notebook reads the Ossie file that Databricks produced from S3 and creates a
semantic view from it. The measure added in Databricks (`TOTAL_QUANTITY`) comes across.

The import is a single built-in function -- Snowflake reads Ossie natively.
No file download or upload needed; both platforms share the same S3 bucket.

## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "EXT_SEMANTIC_INTEROP"
print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

-- Avoid a name conflict with our existing semantic view. In production we would want it to overwrite, but not here.
-- SET target_view = 'SALES_SV_V2';
SET target_view = 'SALES_SV';

SELECT  CONCAT('Imported Semantic View will be titled {{DATABASE}}.{{SCHEMA}}.', $target_view);

## Step 2 - Read the Databricks Ossie from S3

The file is at `s3://<your-bucket>/ossie/ossie_from_databricks.yaml`,
accessible via the external stage.

In [ ]:
%%sql -r dataframe_2
LIST @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE;

## Step 3 - Choose the target view name & load the YAML

Default `SALES_SV_V2` leaves the original `SALES_SV` untouched.

In [ ]:
%%sql -r dataframe_3
SET yaml_content = (
  SELECT $1 FROM @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE/ossie_from_databricks.yaml
  (FILE_FORMAT => '{{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT')
);

SET yaml_to_import = (SELECT REPLACE($yaml_content, 'SALES_SV_V2', $target_view)); -- Translate the semantic view name to prevent conflict (if desired)

SELECT $yaml_to_import;

## Step 4 - Import

In [ ]:
%%sql -r dataframe_7
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('{{DATABASE}}.{{SCHEMA}}', $yaml_to_import);

## Step 5 - Verify

The round-tripped view returns the same numbers as Databricks: EAST 12/750/5, WEST 11/700/5.

In [ ]:
%%sql -r dataframe_8
SET target_fqn = '{{DATABASE}}.{{SCHEMA}}.' || $target_view;
SELECT * FROM SEMANTIC_VIEW(
  IDENTIFIER($target_fqn)
  DIMENSIONS region
  METRICS total_quantity, total_order_amount, order_count
) ORDER BY region;